# Split canónico v1 (seed 42)

Este notebook crea **el único split** que usa todo el equipo. Se corre una vez; escribe `splits/split_v1_seed42.csv` y `.json`, que se suben a git.
Nunca se edita `split_v1_seed42.csv`: si algo cambia, se crea `split_v2_*` con otro notebook.

Decisiones (detalle en `docs/06_SPLIT.md`):

- Los 58 estudios con etiqueta experta (gold) quedan fuera con `fold = -1`. Nunca se entrenan.
- No hay forma de identificar pacientes: `PatientID` es distinto en cada estudio y edad, fechas y peso vienen vacíos.
  Como protección unimos en un mismo grupo los informes largos casi idénticos con el mismo equipo y sexo
  (probablemente la misma persona, por ejemplo rodilla izquierda y derecha).
- 5 folds con `MultilabelStratifiedKFold` sobre grupos. Estratificamos por los positivos de las 12 etiquetas
  (0.5 y NaN cuentan como 0), fabricante, campo magnético e idioma del informe (aproxima el hospital).

In [ ]:
import hashlib, json, os, platform, subprocess
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

RAW = Path("/workspace/data/raw")
LABELS_CSV = Path("/workspace/data/labels/report_labels_v1.csv")
OUT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "knee").exists()) / "splits"
SEED = 42
N_FOLDS = 5
NAME = f"split_v1_seed{SEED}"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA",
          "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

## 1. Metadatos de cada estudio (un DICOM por estudio)

In [ ]:
TAGS = ["PatientID", "PatientSex", "Manufacturer", "ManufacturerModelName", "MagneticFieldStrength", "Laterality"]

def read_tags(study):
    with os.scandir(RAW / "train_series" / study) as it:
        series = next(e.path for e in it if e.is_dir())
    with os.scandir(series) as it:
        path = next(e.path for e in it if e.name.endswith(".dcm"))
    ds = pydicom.dcmread(path, stop_before_pixels=True, specific_tags=TAGS)
    row = {"StudyInstanceUID": study}
    for tag in TAGS:
        value = ds.get(tag, None)
        row[tag] = None if value in (None, "") else str(value)
    return row

train = pd.read_csv(RAW / "train.csv", encoding="utf-8")
with ThreadPoolExecutor(16) as pool:
    meta = pd.DataFrame(list(pool.map(read_tags, train.StudyInstanceUID)))
meta = meta.merge(train[["StudyInstanceUID", "Report"]], on="StudyInstanceUID")
print(len(meta), "estudios;", meta.PatientID.nunique(), "PatientID distintos")

In [ ]:
def vendor(name):
    name = str(name).upper()
    for key, value in [("SIEMENS", "Siemens"), ("PHILIPS", "Philips"), ("GE", "GE")]:
        if name.startswith(key):
            return value
    return "Otro"

def field(value):
    try:
        value = float(value)
    except (TypeError, ValueError):
        return "sin_dato"
    return "3T" if value > 2 else "1.5T" if value > 1.2 else "otro"

def language(text):
    text = str(text).lower()
    if any(w in text for w in ["diz ", "menisküs", "bulgular"]):
        return "tr"
    if any(w in text for w in ["rodilla", "menisco", "técnica"]):
        return "es"
    if any(w in text for w in ["ménisque", "genou"]):
        return "fr"
    if any(w in text for w in ["menisc", "findings", "effusion", "ligament"]):
        return "en"
    return "otro"

meta["vendor"] = meta.Manufacturer.map(vendor)
meta["field"] = meta.MagneticFieldStrength.map(field)
meta["lang"] = meta.Report.map(language)
for column in ["vendor", "field", "lang"]:
    print(column, meta[column].value_counts().to_dict())

## 2. Grupos: informes largos casi idénticos, mismo equipo y sexo

In [ ]:
report = meta.Report.fillna("").str.lower()
normalized = report.str.replace(r"[^a-záéíóúñüçğışöâêèà ]", " ", regex=True).str.split().str.join(" ")
X = TfidfVectorizer(ngram_range=(1, 3)).fit_transform(normalized)
distances, neighbors = NearestNeighbors(n_neighbors=2, metric="cosine").fit(X).kneighbors(X)
similarity = 1 - distances[:, 1]
long_report = meta.Report.fillna("").str.len().to_numpy() > 400

pairs = set()
for a in np.where(long_report & (similarity >= 0.9))[0]:
    b = neighbors[a, 1]
    same_scanner = meta.ManufacturerModelName[a] == meta.ManufacturerModelName[b]
    same_sex = str(meta.PatientSex[a]) == str(meta.PatientSex[b])
    if long_report[b] and same_scanner and same_sex:
        pairs.add(tuple(sorted((int(a), int(b)))))

n = len(meta)
rows = [a for a, b in pairs] + [b for a, b in pairs]
cols = [b for a, b in pairs] + [a for a, b in pairs]
_, component = connected_components(coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(n, n)), directed=False)
meta["group_id"] = component
group_sizes = meta.group_id.value_counts()
print(len(pairs), "pares unidos;", (group_sizes > 1).sum(), "grupos con más de un estudio; máximo", group_sizes.max())

## 3. Gold fuera y folds estratificados

In [ ]:
gold_ids = set(train.loc[train[LABELS].notna().any(axis=1), "StudyInstanceUID"])
meta["is_gold"] = meta.StudyInstanceUID.isin(gold_ids).astype(int)
gold_groups = set(meta.loc[meta.is_gold == 1, "group_id"])

pseudo = pd.read_csv(LABELS_CSV)
positives = pseudo.set_index("StudyInstanceUID")[LABELS].eq(1.0).astype(int)
features = positives.reindex(meta.StudyInstanceUID).fillna(0).astype(int).reset_index(drop=True)
features = pd.concat([features, pd.get_dummies(meta[["vendor", "field", "lang"]]).astype(int)], axis=1)
features["group_id"] = meta.group_id

pool_groups = features.loc[~meta.group_id.isin(gold_groups)].groupby("group_id").max()
splitter = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
group_fold = pd.Series(-1, index=pool_groups.index)
for fold, (_, val) in enumerate(splitter.split(np.zeros(len(pool_groups)), pool_groups.to_numpy())):
    group_fold.iloc[val] = fold

meta["fold"] = meta.group_id.map(group_fold).fillna(-1).astype(int)
meta.loc[meta.group_id.isin(gold_groups) & (meta.is_gold == 0), "fold"] = -2
print(meta.fold.value_counts().sort_index().to_dict())

## 4. Revisiones (si algo falla, el notebook se detiene)

In [ ]:
pool = meta[meta.fold >= 0]
assert meta.StudyInstanceUID.is_unique and len(meta) == len(train)
assert (meta.loc[meta.is_gold == 1, "fold"] == -1).all(), "hay gold dentro de un fold"
assert pool.groupby("group_id").fold.nunique().max() == 1, "un grupo quedó en dos folds"
fold_sizes = pool.fold.value_counts()
assert fold_sizes.max() / fold_sizes.min() < 1.05, f"folds desbalanceados: {fold_sizes.to_dict()}"

pool_pos = positives.reindex(pool.StudyInstanceUID).fillna(0)
pool_pos.index = pool.fold.to_numpy()
balance = pool_pos.groupby(level=0).mean().T
counts = pool_pos.groupby(level=0).sum().T
relative = balance.sub(pool_pos.mean(), axis=0).abs().div(pool_pos.mean(), axis=0)
print("prevalencia de positivos (pseudo) por fold:")
display(balance.round(3))
assert (counts.min(axis=1) >= 20).all(), "alguna etiqueta tiene menos de 20 positivos en un fold"
assert (relative.max(axis=1) < 0.15).all(), f"prevalencia desbalanceada: {relative.max(axis=1).round(3).to_dict()}"
for column in ["vendor", "field", "lang"]:
    display(pd.crosstab(pool.fold, pool[column], normalize="index").round(3))

## 5. Guardar

In [ ]:
columns = ["StudyInstanceUID", "fold", "is_gold", "group_id", "vendor", "field", "lang", "PatientSex", "ManufacturerModelName"]
csv_path = OUT / f"{NAME}.csv"
meta[columns].to_csv(csv_path, index=False)

sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=OUT).stdout.strip()
info = {
    "name": NAME, "seed": SEED, "n_folds": N_FOLDS,
    "method": "MultilabelStratifiedKFold sobre grupos; estratos = 12 positivos pseudo + vendor + field + lang",
    "grouping": "estudio; se unen informes >400 caracteres con coseno TF-IDF >= 0.9, mismo modelo de equipo y sexo",
    "fold_-1": "gold, nunca se entrena", "fold_-2": "estudios agrupados con un gold, fuera de entrenamiento",
    "labels_source": LABELS_CSV.name, "labels_sha256": sha(LABELS_CSV),
    "csv_sha256": sha(csv_path), "git_commit": commit, "python": platform.python_version(),
    "fold_sizes": {int(k): int(v) for k, v in meta.fold.value_counts().sort_index().items()},
    "positives_per_fold": {label: {int(k): int(v) for k, v in row.items()} for label, row in counts.iterrows()},
    "merged_pairs": len(pairs),
}
(OUT / f"{NAME}.json").write_text(json.dumps(info, indent=2, ensure_ascii=False))
print("guardado:", csv_path, info["csv_sha256"][:12])